In [1]:
import pandas as pd
import numpy as np


df_raw = pd.read_csv('user_churn_data.csv') 

df_raw['date'] = pd.to_datetime(df_raw['recency_hours'], unit='h') # 模拟时间对齐


user_first_week = df_raw.groupby('user_id')['date'].min().dt.to_period('W').reset_index()
user_first_week.columns = ['user_id', 'cohort_week']


df_cohort = pd.merge(df_raw, user_first_week, on='user_id')
df_cohort['active_week'] = df_cohort['date'].dt.to_period('W')
df_cohort['week_index'] = (df_cohort['active_week'] - df_cohort['cohort_week']).apply(lambda x: x.n)

cohort_matrix = df_cohort.groupby(['cohort_week', 'week_index'])['user_id'].nunique().unstack().fillna(0)
cohort_size = cohort_matrix.iloc[:, 0]
cohort_retention = cohort_matrix.divide(cohort_size, axis=0)

print("---  Cohort  ---")
print(cohort_retention.round(4) * 100)

---  Cohort  ---
week_index                 0
cohort_week                 
1969-12-29/1970-01-04  100.0
1970-01-05/1970-01-11  100.0


In [3]:
from scipy import stats


group_A = np.random.binomial(1, 0.21, 1000)

group_B = np.random.binomial(1, 0.29, 1000)


t_stat, p_value = stats.ttest_ind(group_A, group_B)

print(f"--- A/B Test ---")
print(f"Control Group (A) Retention Rate: {group_A.mean():.2%}")
print(f"Treatment Group (B) Retention Rate: {group_B.mean():.2%}")
print(f"T-Statistic: {t_stat:.4f} | P-Value: {p_value:.4e}")

if p_value < 0.05:
    print("📢 Business Conclusion: Voucher intervention significantly improved 7-day retention among high-risk users (p < 0.05). Proceed with full roll-out.")
else:
    print("📢 Business Conclusion: No statistically significant difference observed. Insufficient evidence to roll out the voucher policy.")

--- A/B Test ---
Control Group (A) Retention Rate: 21.80%
Treatment Group (B) Retention Rate: 28.20%
T-Statistic: -3.3124 | P-Value: 9.4165e-04
📢 Business Conclusion: Voucher intervention significantly improved 7-day retention among high-risk users (p < 0.05). Proceed with full roll-out.
